In [8]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


In [9]:
base_path   = 'data/repetitions/'
models_path = 'models/'
features    = ['latDev', 'longDev', 'elevSat', 'loraTP', 'loraSF', 'doppler', 'alt', 'raan']


In [10]:
!pwd

/home/ylnner/Documents/new_model/scripts


In [11]:
df = pd.read_csv('baselines_ml_results_v2.csv')
df.head()


,seed,model,accuracy_test,precision_test,recall_test,f1_score_test,auc_roc_test,accuracy_val,precision_val,recall_val,f1_score_val,auc_roc_val
0,0,Logistic Regression,0.804178,0.834423,0.719925,0.772957,0.897692,0.845953,0.822222,0.681579,0.745324,0.942851
1,0,Random Forest,0.888599,0.860714,0.906015,0.882784,0.958694,0.926893,0.868159,0.918421,0.892583,0.979349
2,0,MLP,0.874674,0.877432,0.847744,0.862333,0.957206,0.908616,0.889518,0.826316,0.856753,0.980316
3,1,Logistic Regression,0.858507,0.850365,0.851920,0.851142,0.928512,0.916667,0.924051,0.846868,0.883777,0.978063
4,1,Random Forest,0.912326,0.881849,0.941499,0.910698,0.966835,0.957465,0.926339,0.962877,0.944255,0.989384


In [12]:
# ── Cell 2: Select best seed per model using VALIDATION f1, not test ─────────
# FIX: previously selected seed by test f1, which caused optimistic selection
# bias — test was being used both for selection and final reporting.
best_seeds = (
    df.groupby(['model', 'seed'])['f1_score_val']
    .mean()                         # mean is over 1 row per (model,seed), kept for clarity
    .reset_index()
    .loc[lambda x: x.groupby('model')['f1_score_val'].transform('max') == x['f1_score_val']]
    [['model', 'seed', 'f1_score_val']]
    .rename(columns={'f1_score_val': 'best_val_f1'})
)
 
print("Best seed per model (selected by validation F1):")
print(best_seeds.to_string(index=False))


Best seed per model (selected by validation F1):
              model  seed  best_val_f1
Logistic Regression     3     0.886937
                MLP     3     0.944330
      Random Forest     1     0.944255


In [13]:
# FIX: previously re-trained models from scratch in the notebook, which is
# inconsistent with what was recorded in the CSV.
# FIX: previously re-created and re-fit the scaler, repeating the leakage bug.
final_results = []
 
for _, row in best_seeds.iterrows():
    model_name = row['model']
    seed       = int(row['seed'])
 
    print(f"\n{'='*55}")
    print(f" Model : {model_name}   |   Best seed (by val F1): {seed}")
    print(f"{'='*55}")
 
    # ── Load the exact scaler used during training ────────────────────────────
    scaler = joblib.load(os.path.join(models_path, f'scaler_seed{seed}.pkl'))
 
    # ── Load test data ────────────────────────────────────────────────────────
    path_test = os.path.join(base_path, f'seed_{seed}_final_test_data_transmission.csv')
    df_test   = pd.read_csv(path_test)
    X_test_raw = df_test[features].values
    y_test     = df_test['rcvOk'].values
 
    X_test = scaler.transform(X_test_raw)   # transform only, never re-fit
 
    # ── Load the saved trained model ──────────────────────────────────────────
    model_filename = f"{model_name.replace(' ', '_')}_seed{seed}.pkl"
    model_ml       = joblib.load(os.path.join(models_path, model_filename))
 
    # ── Predict ───────────────────────────────────────────────────────────────
    y_pred = model_ml.predict(X_test)
    y_prob = model_ml.predict_proba(X_test)[:, 1]   # FIX: use proba for AUC-ROC
 
    # ── Report ────────────────────────────────────────────────────────────────
    metrics = {
        "model":     model_name,
        "seed":      seed,
        "accuracy":  accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall":    recall_score(y_test, y_pred, zero_division=0),
        "f1_score":  f1_score(y_test, y_pred, zero_division=0),
        "auc_roc":   roc_auc_score(y_test, y_prob),
    }
    final_results.append(metrics)
 
    for k, v in metrics.items():
        if k not in ("model", "seed"):
            print(f"  {k:<12}: {v:.4f}")
 
 
# ── Cell 4: Final summary table ───────────────────────────────────────────────
df_final = pd.DataFrame(final_results)
print("\nFinal baseline results (best seed per model, selected by val F1):")
df_final



 Model : Logistic Regression   |   Best seed (by val F1): 3
  accuracy    : 0.8714
  precision   : 0.8937
  recall      : 0.8536
  f1_score    : 0.8732
  auc_roc     : 0.9418

 Model : MLP   |   Best seed (by val F1): 3
  accuracy    : 0.9103
  precision   : 0.8989
  recall      : 0.9318
  f1_score    : 0.9150
  auc_roc     : 0.9698

 Model : Random Forest   |   Best seed (by val F1): 1
  accuracy    : 0.9123
  precision   : 0.8818
  recall      : 0.9415
  f1_score    : 0.9107
  auc_roc     : 0.9668

Final baseline results (best seed per model, selected by val F1):


,model,seed,accuracy,precision,recall,f1_score,auc_roc
0,Logistic Regression,3,0.871441,0.893728,0.853577,0.873191,0.941755
1,MLP,3,0.910267,0.898876,0.931780,0.915033,0.969814
2,Random Forest,1,0.912326,0.881849,0.941499,0.910698,0.966835


In [ ]:
# 3. Carga los pesos del archivo .pth
model.load_state_dict(torch.load("bilstm_best_modelo.pth", map_location="cpu"))

# 4. Contar parámetros
params_entrenables = sum(p.numel() for p in model.parameters() if p.requires_grad)
params_totales = sum(p.numel() for p in model.parameters())

print(f"Parámetros Entrenables: {params_entrenables:,}")
print(f"Parámetros Totales:     {params_totales:,}")

In [ ]:
base_path = 'results_server/bilstm/'

for seed in range(num_seeds):
    print(f"--- Procesando Seed {seed} ---")
    
    # Construcción dinámica de nombres de archivos
    path_train = os.path.join(base_path, f'seed_{seed}_final_train_data_transmission.csv')
    path_test  = os.path.join(base_path, f'seed_{seed}_final_test_data_transmission.csv')
    path_val   = os.path.join(base_path, f'seed_{seed}_final_val_data_transmission.csv')

In [9]:
import glob
import os
import torch
from LoraCollisionLSTM import LoraCollisionLSTM


# Getting seed folders
model_files  = sorted(glob.glob(os.path.join("results_server/bilstm", "bilstm_best_seed*")))
device = "cpu"

config = {'hidden_size': 64, 'num_layers': 3, 'dropout': 0.1, 'batch_size': 32}
for mo in model_files:
    print(f"Processing model file: {mo}")
    model_lstm = LoraCollisionLSTM(        
        num_features = 9, # 8 reales + 1 Delta_T (este se adiciona en otras funciones)
        hidden_size  = config['hidden_size'], #64, 
        num_layers   = config['num_layers'], #2, 
        dropout      = config['dropout'] #0.1
    ).to(device)
    
    # 3. Carga los pesos del archivo .pth
    model_lstm.load_state_dict(torch.load(mo, map_location="cpu"))

    # 4. Contar parámetros
    params_entrenables = sum(p.numel() for p in model_lstm.parameters() if p.requires_grad)
    params_totales = sum(p.numel() for p in model_lstm.parameters())

    print(f"Parámetros Entrenables: {params_entrenables:,}")
    print(f"Parámetros Totales:     {params_totales:,}")

Processing model file: results_server/bilstm/bilstm_best_seed0.pth
Parámetros Entrenables: 241,217
Parámetros Totales:     241,217
Processing model file: results_server/bilstm/bilstm_best_seed1.pth
Parámetros Entrenables: 241,217
Parámetros Totales:     241,217
Processing model file: results_server/bilstm/bilstm_best_seed2.pth
Parámetros Entrenables: 241,217
Parámetros Totales:     241,217
Processing model file: results_server/bilstm/bilstm_best_seed3.pth
Parámetros Entrenables: 241,217
Parámetros Totales:     241,217
Processing model file: results_server/bilstm/bilstm_best_seed4.pth
Parámetros Entrenables: 241,217
Parámetros Totales:     241,217
Processing model file: results_server/bilstm/bilstm_best_seed5.pth
Parámetros Entrenables: 241,217
Parámetros Totales:     241,217
Processing model file: results_server/bilstm/bilstm_best_seed6.pth
Parámetros Entrenables: 241,217
Parámetros Totales:     241,217
Processing model file: results_server/bilstm/bilstm_best_seed7.pth
Parámetros Entre

In [ ]:
import glob
import os
import torch
from LoraCollisionTransformer import LoraCollisionTransformer


# Getting seed folders
model_files  = sorted(glob.glob(os.path.join("results_server/transformer", "transformer_best_seed*")))
device = "cpu"

config = {'d_model': 32, 'n_heads': 4, 'n_layers': 3, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 32}
for mo in model_files:
    print(f"Processing model file: {mo}")
    model_transformer = LoraCollisionTransformer(        
        #num_features = 9, # 8 reales + 1 Delta_T (este se adiciona en otras funciones)
        d_model  = config['d_model'], #32,
        n_heads   = config['n_heads'], #4,
        n_layers = config['n_layers'], #3,        
    ).to(device)
    
    # 3. Carga los pesos del archivo .pth
    model_transformer.load_state_dict(torch.load(mo, map_location="cpu"))

    # 4. Contar parámetros
    params_entrenables = sum(p.numel() for p in model_transformer.parameters() if p.requires_grad)
    params_totales = sum(p.numel() for p in model_transformer   .parameters())

    print(f"Parámetros Entrenables: {params_entrenables:,}")
    print(f"Parámetros Totales:     {params_totales:,}")

Processing model file: results_server/transformer/transformer_best_seed0.pth


TypeError: LoraCollisionTransformer.__init__() got an unexpected keyword argument 'num_features'